# Prepare files for steadiercom

In [1]:
import pandas as pd

In [22]:
with open("../data/inorganic.txt") as f:
    mets = f.read().split("\n")

mets_gut = [f"R_EX_{met}__91__e__93__" for met in mets]

with open("../data/Venturelli2018/generated/unlimited.txt",'w') as f:
    text = "\n".join(mets_gut)
    f.write(text)



In [24]:
mets_gut = [f"R_EX_{met}_e" for met in mets]

with open("../../SteadierCom/tests/data/unlimited_R_prefix.txt",'w') as f:
    text = "\n".join(mets_gut)
    f.write(text)


### E.coli mutants

In [2]:
# NB! yhhK in data == panZ in paper
relative_abundance = pd.read_csv("../data/Guillen2021/relative_abundance.csv",index_col=0)
relative_abundance.index =relative_abundance.index.map(lambda x:x if x!="yhhK" else "panZ")

In [3]:
relative_abundance_ecoli = relative_abundance.T.mean().sort_values(ascending=False).head(7)
relative_abundance_ecoli = relative_abundance_ecoli.reset_index()
relative_abundance_ecoli["community"] = relative_abundance_ecoli["index"].map(lambda x:"Ecoli_mutants")
relative_abundance_ecoli["index"] = relative_abundance_ecoli["index"].map(lambda x:"mutant_"+x)
relative_abundance_ecoli\
        .set_index("community")\
        .reset_index()\
        .to_csv("../data/Guillen2021/generated/communities_steadiercom.tsv",
                                                     sep="\t",
                                                     header=None,
                                                     index=None)


### Gut microbiome

**Processing datasetev3**

In [4]:
minimal_medium = pd.read_csv("../output/Venturelli2018/minimal_media.tsv",sep="\t",index_col=0)
member_id2member_name = {x.split("_")[0][0]+x.split("_")[1][0].upper():x for x in minimal_medium.columns}


datasetev3 = pd.read_excel("../data/Venturelli2018/msb178157-sup-0004-datasetev3.xlsx",sheet_name="Dataset EV3")

abundance_dict = {}
last_item = "'ER-'"
abundance_dict[last_item]=[]
for i,row in datasetev3.iterrows():
    list_ = row.iloc[0].split("    ")
    
    if len(list_)==1:
        last_item =list_[0]
        abundance_dict[last_item]=[]
    else:
        abundance_dict[last_item].append(list_[1:])

In [5]:
abundance_dataframes = {community:pd.DataFrame(abundances).map(lambda x: float(x)) for community,abundances in abundance_dict.items()}

for community,dataframe in abundance_dataframes.items(): dataframe.columns=["BH","CA","BU","PC","BO","BV","BT","EL","FP","CH","DP","ER"] 

for community,dataframe in abundance_dataframes.items(): dataframe.columns = dataframe.columns.map(lambda x:member_id2member_name[x])

abundance_series = {community_id.replace("'",""):dataframe.iloc[-1,:] for community_id,dataframe in abundance_dataframes.items()}
for community,series in abundance_series.items(): 
    if community!="NONE": 
        abundance_series[community]=series.drop(member_id2member_name[community[:-1]])


communities_exp_abundance = pd.concat(abundance_series)
communities_exp_abundance

ER-   Blautia_hydrogenotrophica_DSM_10507       0.0109
      Collinsella_aerofaciens_ATCC_25986        0.0255
      Bacteroides_uniformis_ATCC_8492           0.3809
      Prevotella_copri_CB7_DSM_18205            0.0003
      Bacteroides_ovatus_ATCC_8483              0.1343
                                                 ...  
NONE  Eggerthella_lenta_DSM_2243                0.0168
      Faecalibacterium_prausnitzii_A2_165       0.2393
      Clostridium_hiranonis_TO_931_DSM_13275    0.0212
      Desulfovibrio_piger_ATCC_29098            0.0019
      Eubacterium_rectale_ATCC_33656            0.0086
Name: 6, Length: 144, dtype: float64

In [11]:
for community_id in communities_exp_abundance.index.get_level_values(0).unique():
    
    new_id = community_id.replace("-","_minus")
    communities_exp_abundance[[community_id]].to_csv(f"../data/Venturelli2018/generated/communities_steadiercom/{new_id}_abundance.tsv",
                                                     sep="\t",
                                                     header=None)
    
    communities_exp_abundance[[community_id]].reset_index().iloc[:,:-1].to_csv(f"../data/Venturelli2018/generated/communities_steadiercom_no_abundance/{new_id}_abundance.tsv",
                                                     sep="\t",
                                                     header=None,
                                                     index=None)